# Brain Tumor MRI (Nickparvar) - U-Net ResNet34 trained from start to finish on Colab

**What this notebook does, all by itself (you upload nothing):**
install libraries -> download the dataset -> inspect it -> remove duplicate pictures -> split -> train -> test -> Grad-CAM pictures -> save and download the model.

**Before you start:** Runtime -> Change runtime type -> **T4 GPU** -> Save. Then Runtime -> **Run all**, or run the cells from top to bottom.

---
### What I found about this dataset (please read)
- **7,023 MRI pictures, 4 classes:** glioma, meningioma, pituitary, no tumor. Official folders `Training` (5,712) and `Testing` (1,311).
- It is a mix of 3 older datasets (figshare, SARTAJ, Br35H). The "no tumor" pictures come from Br35H. Pictures have different sizes and no patient ID.
- **It has NO segmentation masks.** Nobody drew the tumor outlines. A U-Net learns to draw outlines from masks, so this dataset **cannot teach a U-Net to segment**.
- So this notebook trains the **U-Net ResNet34 model's encoder + classification head** to name the tumor type (4 classes). The U-Net decoder (the part that draws outlines) has nothing to learn from here, so it is not trained.
- To show *where* the model looked, the notebook makes **Grad-CAM heat maps**. They are NOT tumor outlines and must not be called segmentation.
- For real outlines you already have the BRISC model (it has masks).

## 1. Settings (all the knobs are here)

In [ ]:
import os, sys, math, json, time, random, hashlib, shutil, subprocess, zipfile, glob
from pathlib import Path

# ---- settings you may change ----
IMG_SIZE        = 256       # pictures are resized to IMG_SIZE x IMG_SIZE
BATCH_SIZE      = 32
EPOCHS          = 20        # training stops earlier if there is no progress
LEARNING_RATE   = 3e-4
PATIENCE        = 6         # stop if the validation score did not improve for this many epochs
VAL_FRACTION    = 0.15      # part of the Training folder kept for validation
SEED            = 42
USE_PRETRAINED  = True      # ImageNet weights for ResNet34 (downloads about 85 MB)
KAGGLE_DATASET  = "masoudnickparvar/brain-tumor-mri-dataset"
ZENODO_URL      = "https://zenodo.org/records/12735702/files/brain-tumor-mri-dataset.zip?download=1"
WORK_DIR        = "/content/work_braintumor"

# ---- hooks used only to test this notebook outside Colab (ignore them) ----
LOCAL_DATA = os.environ.get("BT_LOCAL_DATA")
EPOCHS = int(os.environ.get("BT_EPOCHS", EPOCHS))
if os.environ.get("BT_NO_PRETRAINED"): USE_PRETRAINED = False
WORK_DIR = os.environ.get("BT_WORK_DIR", WORK_DIR)

os.makedirs(WORK_DIR, exist_ok=True)
os.chdir(WORK_DIR)
print("working folder:", os.getcwd())

## 2. Install the libraries and check them
This cell installs only what is missing, then checks every library. It must end with `ALL LIBRARIES OK`.

In [ ]:
import importlib

NEEDED = {   # import name -> pip name
    "numpy": "numpy", "pandas": "pandas", "PIL": "pillow", "scipy": "scipy", "sklearn": "scikit-learn",
    "matplotlib": "matplotlib", "torch": "torch", "torchvision": "torchvision",
    "segmentation_models_pytorch": "segmentation-models-pytorch", "kagglehub": "kagglehub", "requests": "requests",
}
missing = []
for imp, pip_name in NEEDED.items():
    try:
        importlib.import_module(imp)
    except Exception:
        missing.append(pip_name)
if missing:
    print("installing:", missing)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + missing)
    importlib.invalidate_caches()

bad = []
for imp in NEEDED:
    try:
        module = importlib.import_module(imp)
        print(f"ok   {imp:30s} {getattr(module, '__version__', '')}")
    except Exception as error:
        print(f"MISSING {imp}: {error}")
        bad.append(imp)
print()
print("ALL LIBRARIES OK" if not bad else f"Problem with: {bad}  (run this cell again; if it still fails, restart the runtime)")

## 3. GPU check

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("NO GPU. Runtime -> Change runtime type -> T4 GPU, then run again. (CPU training would be very slow.)")

## 4. Download the dataset
Tries Kaggle first (about 150 MB). If Kaggle refuses, it tries the same dataset on Zenodo (CC BY 4.0).

In [ ]:
import requests, kagglehub

def find_root(folder):
    """the folder that contains both 'Training' and 'Testing' (any upper/lower case)."""
    for base, dirs, _ in os.walk(folder):
        names = {d.lower() for d in dirs}
        if "training" in names and "testing" in names:
            return Path(base)
    return None

DATA_ROOT = None
if LOCAL_DATA:
    DATA_ROOT = find_root(LOCAL_DATA)
if DATA_ROOT is None:
    try:
        print("trying Kaggle ...")
        DATA_ROOT = find_root(kagglehub.dataset_download(KAGGLE_DATASET))
    except Exception as error:
        print("Kaggle download failed:", str(error)[:200])
if DATA_ROOT is None:
    print("trying Zenodo ...")
    target = Path("zenodo_download"); target.mkdir(exist_ok=True)
    zip_path = target / "dataset.zip"
    with requests.get(ZENODO_URL, stream=True, timeout=60) as response:
        response.raise_for_status()
        with open(zip_path, "wb") as f:
            for chunk in response.iter_content(1 << 20):
                f.write(chunk)
    zipfile.ZipFile(zip_path).extractall(target / "files")
    DATA_ROOT = find_root(target / "files")
assert DATA_ROOT is not None, "The dataset was not found. Check the messages above."
print("dataset folder:", DATA_ROOT)

## 5. Look at the dataset (understand it before training)
Counts per class, picture sizes, color modes, and a few example pictures.

In [ ]:
import numpy as np, pandas as pd
from PIL import Image
import matplotlib
import matplotlib.pyplot as plt

EXT = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff"}
rows = []
for split_dir in sorted(p for p in DATA_ROOT.iterdir() if p.is_dir()):
    split = {"training": "Training", "testing": "Testing"}.get(split_dir.name.lower())
    if split is None:
        continue
    for class_dir in sorted(p for p in split_dir.iterdir() if p.is_dir()):
        for f in sorted(class_dir.rglob("*")):
            if f.suffix.lower() in EXT:
                rows.append({"path": str(f), "split": split, "label": class_dir.name.lower()})
df = pd.DataFrame(rows)
assert len(df), "no pictures found"

def head_info(path):
    with Image.open(path) as im:
        return im.size[0], im.size[1], im.mode
info = [head_info(p) for p in df["path"]]
df["w"], df["h"], df["mode"] = zip(*info)
df["size"] = df["w"].astype(str) + "x" + df["h"].astype(str)

CLASSES = sorted(df["label"].unique())
print("classes:", CLASSES)
print("total pictures:", len(df), "\n")
print("pictures per class and split:")
print(pd.crosstab(df["label"], df["split"], margins=True), "\n")
print("color modes (L = grey, RGB = color):")
print(pd.crosstab(df["label"], df["mode"]), "\n")
print("the 3 most common picture sizes per class:")
for c in CLASSES:
    print(f"  {c:12s}", df[df['label'] == c]["size"].value_counts().head(3).to_dict())

EXPECTED = {("Training", "glioma"): 1321, ("Training", "meningioma"): 1339, ("Training", "notumor"): 1595, ("Training", "pituitary"): 1457,
            ("Testing", "glioma"): 300, ("Testing", "meningioma"): 306, ("Testing", "notumor"): 405, ("Testing", "pituitary"): 300}
got = df.groupby(["split", "label"]).size().to_dict()
same = all(got.get(k) == v for k, v in EXPECTED.items())
print("\ncounts match the published table (5712 train / 1311 test):", same)
if not same:
    print("   (not the same as published. Not a problem by itself, but check the table above.)")

# a shortcut check: if one class has a picture size/mode that other classes never have, a model could cheat by learning that
sig = df["size"] + "|" + df["mode"]
for c in CLASSES:
    own = set(sig[df["label"] == c]); others = set(sig[df["label"] != c])
    share = (sig[df["label"] == c].isin(own - others)).mean()
    if share > 0.2:
        print(f"WARNING: {share:.0%} of '{c}' pictures have a size/color-mode that no other class has (shortcut risk).")

fig, axes = plt.subplots(len(CLASSES), 5, figsize=(12, 2.5 * len(CLASSES)))
rng = random.Random(SEED)
for r, c in enumerate(CLASSES):
    picks = rng.sample(list(df[(df["label"] == c) & (df["split"] == "Training")]["path"]), 5)
    for k, p in enumerate(picks):
        axes[r, k].imshow(Image.open(p).convert("L"), cmap="gray"); axes[r, k].axis("off")
        if k == 0: axes[r, k].set_title(c, loc="left", fontsize=10)
plt.tight_layout(); plt.savefig("examples.png", dpi=90); plt.show()

## 6. Remove duplicate pictures and make the validation split
This dataset was built from 3 public sets, so the same picture can appear twice. If a picture is in both Training and Testing, the test score looks better than it really is.

- A picture counts as a copy when, shrunk to 64x64, it is almost identical to another one (correlation > 0.998 and tiny pixel difference). Small JPEG or resize differences are ignored. I checked this rule on real brain MRI pictures: re-saved and resized copies were caught, and different brains were not.
- Copies found in **Testing** are removed from **Training** (Testing stays untouched).
- Copies inside Training are reduced to one.
- Safety: if more than 15% of Training would be removed, the notebook thinks the rule is wrong, prints a warning and removes nothing.

In [ ]:
def small_gray(path, n=64):
    with Image.open(path) as im:
        return np.asarray(im.convert("L").resize((n, n), Image.BILINEAR), dtype=np.float32)

small = np.stack([small_gray(p) for p in df["path"]])                 # (N, 64, 64)
flat = small.reshape(len(small), -1)
zs = flat - flat.mean(axis=1, keepdims=True)
zs /= np.linalg.norm(zs, axis=1, keepdims=True) + 1e-8                # for the correlation between pictures

CORR_MIN, MAD_MAX = 0.998, 1.5       # two pictures are copies if correlation > 0.998 AND mean difference < 1.5 (scale 0-255)
parent = list(range(len(df)))        # (tested on real MRI: re-saved/resized copies give correlation >= 0.9996 and difference <= 0.6)
def find(a):
    while parent[a] != a:
        parent[a] = parent[parent[a]]; a = parent[a]
    return a

pairs = 0
for start in range(0, len(df), 500):
    corr = zs[start:start + 500] @ zs.T
    for i, j in zip(*np.where(corr > CORR_MIN)):
        i += start
        if j <= i:
            continue
        if np.abs(flat[i] - flat[j]).mean() < MAD_MAX:
            parent[find(i)] = find(j); pairs += 1
df["group"] = [find(i) for i in range(len(df))]
group_size = df.groupby("group")["path"].transform("size")
print("copy pairs found:", pairs, "| pictures that have at least one copy:", int((group_size > 1).sum()))

group_has_test = df.groupby("group")["split"].transform(lambda s: bool((s == "Testing").any()))
leak = (df["split"] == "Training") & group_has_test                    # Training pictures that also exist in Testing
train_only = df[(df["split"] == "Training") & ~group_has_test]
extra = train_only.index[train_only.duplicated("group", keep="first")]  # 2nd, 3rd ... copy inside Training
drop_idx = set(df.index[leak]) | set(extra)
train_n = int((df["split"] == "Training").sum())
print(f"Training pictures that are also in Testing: {int(leak.sum())}")
print(f"extra copies inside Training: {len(extra)}")
if len(drop_idx) > 0.15 * train_n:
    print(f"WARNING: {len(drop_idx)} pictures ({len(drop_idx) / train_n:.0%}) would be removed. That is too many, the rule may be wrong. Nothing removed.")
    drop_idx = set()
else:
    print(f"removed from Training: {len(drop_idx)}")
clean = df.drop(index=list(drop_idx)).reset_index(drop=True)

# stratified validation split from the cleaned Training pictures (Testing is not touched)
rng = np.random.RandomState(SEED)
clean["part"] = "test"
for c in CLASSES:
    idx = clean.index[(clean["split"] == "Training") & (clean["label"] == c)].to_numpy().copy()
    rng.shuffle(idx)
    n_val = int(round(len(idx) * VAL_FRACTION))
    clean.loc[idx[:n_val], "part"] = "val"
    clean.loc[idx[n_val:], "part"] = "train"
clean.drop(columns=["group"]).to_csv("clean_split.csv", index=False)
print()
print(pd.crosstab(clean["label"], clean["part"], margins=True))
print("\nNote: the dataset has no patient ID, so pictures of the same person may still be in train, validation and test.")

## 7. Load all pictures into memory
Pictures are resized to the same square size and kept as small integers, so training is fast.

In [ ]:
def load_resized(path):
    with Image.open(path) as im:
        return np.array(im.convert("RGB").resize((IMG_SIZE, IMG_SIZE), Image.BILINEAR), dtype=np.uint8)

class_to_idx = {c: i for i, c in enumerate(CLASSES)}
data = {}
t0 = time.time()
for part in ("train", "val", "test"):
    sub = clean[clean["part"] == part]
    X = np.stack([load_resized(p) for p in sub["path"]])
    y = np.array([class_to_idx[l] for l in sub["label"]], dtype=np.int64)
    data[part] = (torch.from_numpy(X).permute(0, 3, 1, 2).contiguous(), torch.from_numpy(y))
    print(f"{part:5s} {tuple(data[part][0].shape)}  {time.time() - t0:.0f}s")

## 8. The model: U-Net with a ResNet34 encoder
`segmentation_models_pytorch` builds a U-Net. Here we use its **ResNet34 encoder** and its **classification head**, because this dataset only has class labels (no masks). The decoder is not used.

In [ ]:
import torch.nn as nn, torch.nn.functional as F
import segmentation_models_pytorch as smp

def build_model(pretrained):
    return smp.Unet(encoder_name="resnet34", encoder_weights="imagenet" if pretrained else None, in_channels=3,
                    classes=1, aux_params={"classes": len(CLASSES), "pooling": "avg", "dropout": 0.3})

model = build_model(USE_PRETRAINED).to(device)

def classify(model, x):
    """encoder -> classification head (the decoder is skipped)."""
    return model.classification_head(model.encoder(x)[-1])

MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
def normalize(x01):
    return (x01 - MEAN) / STD

n_params = sum(p.numel() for p in list(model.encoder.parameters()) + list(model.classification_head.parameters()))
print(f"classes: {CLASSES}")
print(f"trained part (encoder + head): {n_params / 1e6:.1f} M parameters, pretrained ImageNet weights: {USE_PRETRAINED}")

## 9. Training
- Augmentation (on the GPU): small rotation, zoom, shift, left-right flip, brightness/contrast change. No up-down flip.
- Loss: cross-entropy with class weights and a little label smoothing.
- The best epoch is chosen by **macro F1 on the validation pictures**. The Testing pictures are NOT used here.

In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def augment(x):
    """x: (B,3,H,W) floats 0..1 on the device."""
    B = x.size(0)
    r = lambda: torch.rand(B, device=x.device) * 2 - 1
    angle, scale = r() * math.radians(10), 1 + r() * 0.10
    flip = (torch.rand(B, device=x.device) < 0.5).float() * 2 - 1
    cos, sin = torch.cos(angle) / scale, torch.sin(angle) / scale
    theta = torch.stack([torch.stack([flip * cos, -sin, r() * 0.05], 1), torch.stack([flip * sin, cos, r() * 0.05], 1)], 1)
    grid = F.affine_grid(theta, list(x.shape), align_corners=False)
    x = F.grid_sample(x, grid, mode="bilinear", padding_mode="zeros", align_corners=False)
    x = x * (1 + r().view(B, 1, 1, 1) * 0.2) + r().view(B, 1, 1, 1) * 0.05
    return x.clamp(0, 1)

def batches(X, y, size, shuffle):
    order = torch.randperm(len(X)) if shuffle else torch.arange(len(X))
    for i in range(0, len(X), size):
        idx = order[i:i + size]
        yield X[idx].to(device, non_blocking=True).float() / 255, y[idx].to(device)

@torch.no_grad()
def predict_all(model, X, y):
    model.eval()
    logits_all = []
    for xb, _ in batches(X, y, BATCH_SIZE * 2, False):
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            logits_all.append(classify(model, normalize(xb)).float().cpu())
    return torch.cat(logits_all)

counts = torch.bincount(data["train"][1], minlength=len(CLASSES)).float()
class_weights = (counts.sum() / (len(CLASSES) * counts)).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
params = list(model.encoder.parameters()) + list(model.classification_head.parameters())
optimizer = torch.optim.AdamW(params, lr=LEARNING_RATE, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)
scaler = torch.amp.GradScaler("cuda", enabled=device.type == "cuda")

os.makedirs("models", exist_ok=True)
MODEL_PATH = "models/braintumor_unet_resnet34_cls.pt"
log, best, bad_epochs = [], -1.0, 0
Xtr, ytr = data["train"]
for epoch in range(1, EPOCHS + 1):
    t0 = time.time(); model.train(); total, seen = 0.0, 0
    for xb, yb in batches(Xtr, ytr, BATCH_SIZE, True):
        xb = normalize(augment(xb))
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=device.type == "cuda"):
            loss = criterion(classify(model, xb).float(), yb)
        scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update()
        total += loss.item() * len(yb); seen += len(yb)
    val_logits = predict_all(model, *data["val"])
    val_pred, val_true = val_logits.argmax(1).numpy(), data["val"][1].numpy()
    val_loss = F.cross_entropy(val_logits, data["val"][1]).item()
    val_acc, val_f1 = accuracy_score(val_true, val_pred), f1_score(val_true, val_pred, average="macro")
    scheduler.step(val_f1)
    log.append({"epoch": epoch, "train_loss": total / seen, "val_loss": val_loss, "val_acc": val_acc, "val_f1": val_f1,
                "lr": optimizer.param_groups[0]["lr"], "seconds": time.time() - t0})
    mark = ""
    if val_f1 > best:
        best, bad_epochs, mark = val_f1, 0, "  <- saved best model"
        torch.save({"state_dict": model.state_dict(), "classes": CLASSES, "img_size": IMG_SIZE, "epoch": epoch,
                    "val_f1": float(val_f1), "val_acc": float(val_acc)}, MODEL_PATH)
    else:
        bad_epochs += 1
    print(f"epoch {epoch:2d}/{EPOCHS}  train loss {total / seen:.3f}  val loss {val_loss:.3f}  val acc {val_acc:.3f}  val F1 {val_f1:.3f}  {time.time() - t0:.0f}s{mark}")
    if bad_epochs >= PATIENCE:
        print(f"no improvement for {PATIENCE} epochs, stopping early.")
        break

log_df = pd.DataFrame(log); log_df.to_csv("training_log.csv", index=False)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(log_df["epoch"], log_df["train_loss"], label="train"); ax[0].plot(log_df["epoch"], log_df["val_loss"], label="validation")
ax[0].set_title("loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(log_df["epoch"], log_df["val_acc"], label="accuracy"); ax[1].plot(log_df["epoch"], log_df["val_f1"], label="macro F1")
ax[1].set_title("validation score"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.tight_layout(); plt.savefig("training_curves.png", dpi=90); plt.show()
print(f"\nbest validation macro F1: {best:.3f}   model saved: {MODEL_PATH}")

## 10. Final test (the official Testing folder, never used for training or choosing)

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
ckpt = torch.load(MODEL_PATH, map_location=device)
model.load_state_dict(ckpt["state_dict"])
test_logits = predict_all(model, *data["test"])
y_true, y_pred = data["test"][1].numpy(), test_logits.argmax(1).numpy()
probs = torch.softmax(test_logits, 1).numpy()

acc, macro_f1 = accuracy_score(y_true, y_pred), f1_score(y_true, y_pred, average="macro")
report = classification_report(y_true, y_pred, target_names=CLASSES, output_dict=True, zero_division=0)
cm = confusion_matrix(y_true, y_pred, labels=range(len(CLASSES)))

print(f"model from epoch {ckpt['epoch']} (validation macro F1 {ckpt['val_f1']:.3f})")
print(f"\nTEST RESULTS ({len(y_true)} pictures)")
print(f"accuracy: {acc:.3f}     macro F1: {macro_f1:.3f}\n")
print(pd.DataFrame(report).T.loc[CLASSES + ["macro avg"], ["precision", "recall", "f1-score", "support"]].round(3).to_string())
print("\nconfusion matrix (rows = real class, columns = what the model said):")
print(pd.DataFrame(cm, index=[f"real {c}" for c in CLASSES], columns=[f"said {c}" for c in CLASSES]).to_string())

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.imshow(cm, cmap="Blues")
ax.set_xticks(range(len(CLASSES))); ax.set_xticklabels(CLASSES, rotation=30, ha="right")
ax.set_yticks(range(len(CLASSES))); ax.set_yticklabels(CLASSES)
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, int(cm[i, j]), ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
ax.set_xlabel("model said"); ax.set_ylabel("real"); plt.tight_layout(); plt.savefig("confusion_matrix.png", dpi=90); plt.show()

json.dump({"n_test": int(len(y_true)), "accuracy": acc, "macro_f1": macro_f1, "classes": CLASSES, "per_class": {c: report[c] for c in CLASSES},
           "confusion": cm.tolist(), "checkpoint_epoch": ckpt["epoch"], "removed_train_duplicates": len(drop_idx),
           "img_size": IMG_SIZE, "note": "no patient id in the dataset, real-life scores can be lower"},
          open("test_results.json", "w"), indent=2)
print("\nsaved: test_results.json, confusion_matrix.png")

## 11. Grad-CAM: where did the model look?
Red/yellow = the area that pushed the model toward its answer. **This is NOT a tumor outline and NOT a segmentation.** It only shows where the model looked, and it can look at the wrong place even when the answer is right.

In [ ]:
def grad_cam(model, x01, class_index=None):
    """x01: (1,3,H,W) floats 0..1. returns (heat map HxW in 0..1, predicted class, probabilities)."""
    model.eval()
    store = {}
    layer = model.encoder.layer4
    h1 = layer.register_forward_hook(lambda m, i, o: store.__setitem__("a", o))
    h2 = layer.register_full_backward_hook(lambda m, gi, go: store.__setitem__("g", go[0]))
    try:
        with torch.enable_grad():
            x = normalize(x01.to(device)).requires_grad_(True)
            logits = classify(model, x)
            idx = int(logits.argmax(1)) if class_index is None else class_index
            model.zero_grad(); logits[0, idx].backward()
        w = store["g"].mean(dim=(2, 3), keepdim=True)
        cam = F.relu((w * store["a"]).sum(1, keepdim=True))
        cam = F.interpolate(cam, size=x01.shape[-2:], mode="bilinear", align_corners=False)[0, 0].detach().cpu().numpy()
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam, idx, torch.softmax(logits.detach(), 1)[0].cpu().numpy()
    finally:
        h1.remove(); h2.remove()

Xte, yte = data["test"]
right = np.where(y_pred == y_true)[0]; wrong = np.where(y_pred != y_true)[0]
rng = np.random.RandomState(SEED)
show = list(rng.choice(right, min(4, len(right)), replace=False)) + list(rng.choice(wrong, min(4, len(wrong)), replace=False))
cols = 4; rows_n = math.ceil(len(show) / cols)
fig, axes = plt.subplots(rows_n, cols, figsize=(3.2 * cols, 3.9 * rows_n)); axes = np.atleast_2d(axes)
for ax in axes.ravel(): ax.axis("off")
for ax, i in zip(axes.ravel(), show):
    x01 = (Xte[i:i + 1].float() / 255)
    cam, pred, p = grad_cam(model, x01)
    ax.imshow(x01[0].permute(1, 2, 0).numpy()); ax.imshow(cam, cmap="jet", alpha=0.4)
    ok = pred == int(yte[i])
    ax.set_title(f"real: {CLASSES[int(yte[i])]}\nsaid: {CLASSES[pred]} {p[pred]:.0%}", fontsize=9, color="green" if ok else "red")
plt.suptitle("Grad-CAM (where the model looked, NOT a tumor outline)"); plt.tight_layout(rect=[0, 0, 1, 0.95]); plt.subplots_adjust(hspace=0.35)
plt.savefig("gradcam_examples.png", dpi=90); plt.show()

## 12. Try the model on one picture
After training you can call `predict_image("path/to/picture.jpg")`. By default it uses one random test picture.

In [ ]:
def predict_image(path):
    x01 = torch.from_numpy(load_resized(path)).permute(2, 0, 1).unsqueeze(0).float() / 255
    cam, pred, p = grad_cam(model, x01)
    print("prediction:", CLASSES[pred], f"({p[pred]:.1%})")
    for c, v in zip(CLASSES, p):
        print(f"  {c:12s} {v:.1%}")
    plt.figure(figsize=(4, 4)); plt.imshow(x01[0].permute(1, 2, 0).numpy()); plt.imshow(cam, cmap="jet", alpha=0.4); plt.axis("off"); plt.show()

predict_image(clean[clean["part"] == "test"].sample(1, random_state=SEED)["path"].iloc[0])

## 13. Save everything and download
Makes one zip with the model, scores, curves and the clean split list. It is also copied to your Google Drive if you allow it (Colab deletes everything when it disconnects).

In [ ]:
pack = Path("pack"); pack.mkdir(exist_ok=True)
for f in ["models/braintumor_unet_resnet34_cls.pt", "test_results.json", "training_log.csv", "training_curves.png",
          "confusion_matrix.png", "gradcam_examples.png", "examples.png", "clean_split.csv"]:
    if os.path.exists(f):
        target = pack / Path(f).name
        shutil.copy(f, target)
shutil.make_archive("braintumor_results", "zip", "pack")
print("created:", os.path.abspath("braintumor_results.zip"), f"({os.path.getsize('braintumor_results.zip') / 1e6:.0f} MB)")

try:
    from google.colab import drive, files
    try:
        drive.mount("/content/drive")
        shutil.copy("braintumor_results.zip", "/content/drive/MyDrive/braintumor_results.zip")
        print("copy saved to Google Drive: braintumor_results.zip")
    except Exception as error:
        print("Drive copy skipped:", str(error)[:150])
    files.download("braintumor_results.zip")
except ImportError:
    print("(not running in Colab: nothing to download)")